1. Getting dataset from parquet file and describing Dtypes

In [82]:
import pandas as pd
df_ex=pd.read_parquet('../data/raw_data.parquet')
df_ex.info()


<class 'pandas.DataFrame'>
RangeIndex: 539326 entries, 0 to 539325
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id_transaccion  539326 non-null  str    
 1   id_vehiculo     539324 non-null  float64
 2   id_ubicacion    539325 non-null  float64
 3   id_cliente      539325 non-null  float64
 4   fecha           539326 non-null  str    
 5   unidades        539326 non-null  int64  
 6   venta_bruta     523160 non-null  float64
 7   marca           498457 non-null  str    
 8   modelo          498457 non-null  str    
dtypes: float64(4), int64(1), str(4)
memory usage: 53.9 MB


2. Changing Dtypes for date and id_vehiculo, id_cliente and id_ubicacion

In [83]:
df_ex['fecha']=pd.to_datetime(df_ex['fecha'])
df_ex[['id_vehiculo','id_ubicacion','id_cliente']] = df_ex[['id_vehiculo','id_ubicacion','id_cliente']].astype('Int64')

In [84]:
df_ex.head(10)

,id_transaccion,id_vehiculo,id_ubicacion,id_cliente,fecha,unidades,venta_bruta,marca,modelo
0,ST-2000000,1,1,2536,2024-12-01,1,679237.0,Alfa Romeo,Giulia
1,ST-2000001,2,2,4867,2022-09-22,1,419582.0,Alfa Romeo,Stelvio
2,ST-2000002,3,1,4669,2024-12-01,1,834373.0,Alfa Romeo,Tonale
3,ST-2000003,4,3,899,2024-04-27,1,945477.0,Dodge,Attitude
4,ST-2000004,5,4,2471,2025-11-13,1,433661.0,Dodge,Durango
5,ST-2000005,6,4,6717,2022-01-06,1,428597.0,Dodge,Journey
6,ST-2000006,7,1,3359,2024-05-07,1,1065849.0,Fiat,Argo
7,ST-2000007,1,4,3313,2024-06-07,1,375204.0,Alfa Romeo,Giulia
8,ST-2000008,8,1,2377,2023-06-16,1,468735.0,Fiat,Ducato
9,ST-2000009,4,2,1750,2022-05-16,1,802080.0,Dodge,Attitude


In [85]:
df_ex.info()

<class 'pandas.DataFrame'>
RangeIndex: 539326 entries, 0 to 539325
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id_transaccion  539326 non-null  str           
 1   id_vehiculo     539324 non-null  Int64         
 2   id_ubicacion    539325 non-null  Int64         
 3   id_cliente      539325 non-null  Int64         
 4   fecha           539326 non-null  datetime64[us]
 5   unidades        539326 non-null  int64         
 6   venta_bruta     523160 non-null  float64       
 7   marca           498457 non-null  str           
 8   modelo          498457 non-null  str           
dtypes: Int64(3), datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 50.3 MB


3. Checking out nulls

In [86]:
serie_null = df_ex.isnull().sum()
print(type(serie_null))
print(serie_null)

<class 'pandas.Series'>
id_transaccion        0
id_vehiculo           2
id_ubicacion          1
id_cliente            1
fecha                 0
unidades              0
venta_bruta       16166
marca             40869
modelo            40869
dtype: int64


3.1 Calculating percentages for null values on each column

In [87]:
total_rows = df_ex['id_transaccion'].count()
valores = []
for i in range(0, len(serie_null)):
    valores.append(serie_null.iloc[i]/total_rows * 100)

In [88]:
print(type(valores))
print(valores)

<class 'list'>
[0.0, 0.00037083322517364264, 0.00018541661258682132, 0.00018541661258682132, 0.0, 0.0, 2.9974449590785537, 7.577791539810801, 7.577791539810801]


4. Cleansing null values, we evaluate, if we har less than 15% of missing values we can proceed to delete them

In [89]:
df_ex = df_ex.dropna()
df_ex.info()



<class 'pandas.DataFrame'>
Index: 483460 entries, 0 to 539320
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id_transaccion  483460 non-null  str           
 1   id_vehiculo     483460 non-null  Int64         
 2   id_ubicacion    483460 non-null  Int64         
 3   id_cliente      483460 non-null  Int64         
 4   fecha           483460 non-null  datetime64[us]
 5   unidades        483460 non-null  int64         
 6   venta_bruta     483460 non-null  float64       
 7   marca           483460 non-null  str           
 8   modelo          483460 non-null  str           
dtypes: Int64(3), datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 49.3 MB


In [90]:
df_ex.isnull().sum()

id_transaccion    0
id_vehiculo       0
id_ubicacion      0
id_cliente        0
fecha             0
unidades          0
venta_bruta       0
marca             0
modelo            0
dtype: int64

5. Checking out duplicated values

In [91]:
df_ex.apply(lambda col: col.duplicated().sum())

id_transaccion      2580
id_vehiculo       483437
id_ubicacion      483453
id_cliente        468028
fecha             482116
unidades          483457
venta_bruta       143401
marca             483454
modelo            483437
dtype: int64

5.1 To get a star schema we need to avoid duplicated values on 'id_transaccion' primary key

In [92]:
id_trans_dup = df_ex['id_transaccion'].duplicated().sum()
print(type(id_trans_dup))
print(id_trans_dup)

<class 'numpy.int64'>
2580


5.2 As we have only a few duplicated values we can proceed to delete them

In [93]:
N_duplicados = id_trans_dup / df_ex['id_transaccion'].count() * 100 
print(f'{round(N_duplicados,1)}% de valores duplicados')

0.5% de valores duplicados


In [94]:
df_ex = df_ex.drop_duplicates(subset='id_transaccion', keep='first')
df_ex = df_ex.reset_index(drop=True)

6. We change an specific stakeholder requirement: Change Ram to RAM if exists any

In [95]:
df_ex['marca'] = df_ex['marca'].str.title().replace({'Ram':'RAM'})

6.1 We check brand counting, we expect to have only six different 

In [96]:
conteo_marcas = df_ex['marca'].unique()
print(type(conteo_marcas))
conteo_marcas

<class 'pandas.arrays.ArrowStringArray'>


<ArrowStringArray>
['Alfa Romeo', 'Dodge', 'Fiat', 'Jeep', 'Peugeot', 'RAM']
Length: 6, dtype: str

6.2 We check model counting, we expect to have only twentythree different 

In [97]:
conteo_modelos = df_ex['modelo'].unique()
print(type(conteo_modelos))
conteo_modelos

<class 'pandas.arrays.ArrowStringArray'>


<ArrowStringArray>
[        'Giulia',        'Stelvio',         'Tonale',       'Attitude',
        'Durango',        'Journey',           'Argo',         'Ducato',
           'Mobi',          'Pulse',        'Compass',      'Gladiator',
 'Grand Cherokee',       'Renegade',       'Wrangler',           '2008',
           '3008',           '5008',        'Manager',           '1500',
           '2500',            '700',      'Promaster']
Length: 23, dtype: str

In [98]:
df_ex.duplicated().sum()
df_ex.info()

<class 'pandas.DataFrame'>
RangeIndex: 480880 entries, 0 to 480879
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id_transaccion  480880 non-null  str           
 1   id_vehiculo     480880 non-null  Int64         
 2   id_ubicacion    480880 non-null  Int64         
 3   id_cliente      480880 non-null  Int64         
 4   fecha           480880 non-null  datetime64[us]
 5   unidades        480880 non-null  int64         
 6   venta_bruta     480880 non-null  float64       
 7   marca           480880 non-null  str           
 8   modelo          480880 non-null  str           
dtypes: Int64(3), datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 45.3 MB


7. Sub set creation, transactions above $600,000 MXN

In [99]:
estadisticos_1 = df_ex.describe()
print(type(estadisticos_1))
print(estadisticos_1)

<class 'pandas.DataFrame'>
       id_vehiculo  id_ubicacion   id_cliente                       fecha  \
count     480880.0      480880.0     480880.0                      480880   
mean      7.749193      3.248216  7704.477982  2024-01-26 07:43:40.735318   
min            1.0           1.0          1.0         2022-01-01 00:00:00   
25%            2.0           1.0       3844.0         2023-04-17 00:00:00   
50%            5.0           4.0       7705.0         2023-12-10 00:00:00   
75%           13.0           4.0      11563.0         2024-11-11 00:00:00   
max           23.0           7.0      15432.0         2025-12-28 00:00:00   
std       6.044193      1.756427  4455.518408                         NaN   

            unidades   venta_bruta  
count  480880.000000  4.808800e+05  
mean        1.035121  6.775157e+05  
min         1.000000  3.400010e+05  
25%         1.000000  4.978455e+05  
50%         1.000000  6.551915e+05  
75%         1.000000  8.121815e+05  
max         5.000000

In [100]:
average_1 = df_ex['venta_bruta'].mean()
print(type(average_1))
print(average_1)

<class 'numpy.float64'>
677515.7367014639


In [101]:
df_sub = df_ex.loc[df_ex['venta_bruta']>average_1,['id_transaccion','venta_bruta','marca']]
print(df_sub.head(10))
print(type(df_sub))
print(len(df_sub))

   id_transaccion  venta_bruta       marca
0      ST-2000000     679237.0  Alfa Romeo
2      ST-2000002     834373.0  Alfa Romeo
3      ST-2000003     945477.0       Dodge
6      ST-2000006    1065849.0        Fiat
9      ST-2000009     802080.0       Dodge
12     ST-2000012     790998.0        Fiat
13     ST-2000013     904146.0        Jeep
15     ST-2000015     763342.0  Alfa Romeo
16     ST-2000016     814619.0        Jeep
17     ST-2000017     888967.0        Jeep
<class 'pandas.DataFrame'>
223191


Day 2 practice

In [102]:
df_ex['anio'] = df_ex['fecha'].dt.year
df_ex['mes'] = df_ex['fecha'].dt.month_name()
df_ex['dia'] = df_ex['fecha'].dt.day_name()
df_ex.head(10)

,id_transaccion,id_vehiculo,id_ubicacion,id_cliente,fecha,unidades,venta_bruta,marca,modelo,anio,mes,dia
0,ST-2000000,1,1,2536,2024-12-01,1,679237.0,Alfa Romeo,Giulia,2024,December,Sunday
1,ST-2000001,2,2,4867,2022-09-22,1,419582.0,Alfa Romeo,Stelvio,2022,September,Thursday
2,ST-2000002,3,1,4669,2024-12-01,1,834373.0,Alfa Romeo,Tonale,2024,December,Sunday
3,ST-2000003,4,3,899,2024-04-27,1,945477.0,Dodge,Attitude,2024,April,Saturday
4,ST-2000004,5,4,2471,2025-11-13,1,433661.0,Dodge,Durango,2025,November,Thursday
5,ST-2000005,6,4,6717,2022-01-06,1,428597.0,Dodge,Journey,2022,January,Thursday
6,ST-2000006,7,1,3359,2024-05-07,1,1065849.0,Fiat,Argo,2024,May,Tuesday
7,ST-2000007,1,4,3313,2024-06-07,1,375204.0,Alfa Romeo,Giulia,2024,June,Friday
8,ST-2000008,8,1,2377,2023-06-16,1,468735.0,Fiat,Ducato,2023,June,Friday
9,ST-2000009,4,2,1750,2022-05-16,1,802080.0,Dodge,Attitude,2022,May,Monday


In [103]:
df_cleaned = df_ex
df_cleaned.to_parquet('../data/cl_data.parquet', index=False)